# 03 | Practice Solutions: Mystery Solved

**Amity University, Lucknow | MBA | 9 October 2026**

![Python](https://img.shields.io/badge/Python-3-3776AB?logo=python&logoColor=white)
![NumPy](https://img.shields.io/badge/NumPy-Arrays-013243?logo=numpy&logoColor=white)
![Pandas](https://img.shields.io/badge/Pandas-Analytics-150458?logo=pandas&logoColor=white)
![Audience](https://img.shields.io/badge/Audience-MBA_students-00897B)
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zaid-kamil/Data-analytics-using-numpy-and-python-at-Amity-Lucknow/blob/main/solutions/04_practice_solutions.ipynb)

### Your instructor: Zaid Kamil



## Your brief
Analyse two fictional datasets: retail transactions and marketing campaigns. All monetary values are INR. Work through the prompts in order, using the earlier notebooks for help. Allow 30–45 minutes for self-study; choose one case for a shorter class exercise.

Retail gross profit excludes overheads and taxes. Marketing revenue is attributed revenue; these records do not prove incremental impact. Missing values and zero-result campaigns are intentional—the datasets have brought their own troublemakers.


## The analysis journey

Follow the same route for both datasets. Skipping directly to *Recommend* is tempting, but the duplicate row is waiting to cause chaos.

![Six-step data analysis workflow](../assets/diagrams/analysis-workflow.svg)


## Load your practice data
Run this cell first. It reads local CSVs or fetches the repository copies in Colab. Colab therefore needs internet access. No data, no detective work.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

def load_dataset(filename):
    # Works from the repository root, notebooks/, or solutions/.
    for folder in (Path('datasets'), Path('../datasets')):
        path = folder / filename
        if path.exists():
            return pd.read_csv(path)
    # Colab opens a notebook without cloning its supporting files.
    base = 'https://raw.githubusercontent.com/zaid-kamil/Data-analytics-using-numpy-and-python-at-Amity-Lucknow/main/datasets/'
    return pd.read_csv(base + filename)

retail_raw = load_dataset('sales.csv')
marketing_raw = load_dataset('campaigns.csv')
print('Retail rows:', len(retail_raw))
print('Marketing rows:', len(marketing_raw))


Retail rows: 20
Marketing rows: 20


## 1. Meet the datasets before trusting them

Display the first five rows, data types, missing-value counts, and duplicate counts. Which fields could change a business conclusion if incomplete? Never trust a table before a proper introduction.


In [2]:
print(retail_raw.head())
print(marketing_raw.head())
for label, frame in [('Retail', retail_raw), ('Marketing', marketing_raw)]:
    print(label)
    print(frame.dtypes)
    print(frame.isna().sum())
    print('Exact duplicate rows:', frame.duplicated().sum())


  Order_ID        Date   Product   Region  Units  Unit_Price  Unit_Cost
0     R001  2026-09-01  Notebook  Lucknow   20.0         100         60
1     R002  2026-09-02       Pen    Delhi   50.0          20         10
2     R003  2026-09-03   Planner   Kanpur    8.0         300        180
3     R004  2026-09-04  Notebook    Delhi   30.0         100         60
4     R005  2026-09-05   Planner  Lucknow   12.0         300        180
  Campaign_ID Channel    Spend  Impressions  Clicks  Leads  Customers  Revenue
0        C001  Search  12000.0        50000    2500    200         40    40000
1        C002  Social   8000.0        80000    1600    120         20    18000
2        C003   Email   2000.0        12000    1800    150         35    28000
3        C004  Search  15000.0        60000    3000    240         45    45000
4        C005  Social  10000.0       100000    2000    140         25    22000
Retail
Order_ID          str
Date              str
Product           str
Region            str

## 2. Clean the retail data

Assume one row per Order_ID. Remove repeated IDs, save incomplete Units rows separately, and exclude them from the analysis. Convert Date to datetime and Units to integers. Do not invent missing sales—the spreadsheet already has enough imagination.


In [3]:
retail_unique = retail_raw.drop_duplicates('Order_ID').copy()
retail_review = retail_unique.loc[retail_unique['Units'].isna()].copy()
retail = retail_unique.dropna(subset=['Units']).copy()
retail['Date'] = pd.to_datetime(retail['Date'])
retail['Units'] = retail['Units'].astype(int)
print('Usable rows:', len(retail), '| Review rows:', len(retail_review))


Usable rows: 18 | Review rows: 1


## 3. Calculate retail metrics

Create Revenue, Total_Cost, Gross_Profit, and Margin_Pct. Unit_Price and Unit_Cost are per unit. Show overall revenue, gross profit, and weighted margin. This is where rows finally turn into business answers.


### How the retail numbers connect

The same `Units` value helps calculate both revenue and cost. Profit is what remains after the business pays for the products—sadly, revenue does not all get to stay.

![Retail revenue, cost, and gross profit formulas](../assets/diagrams/retail-metrics.svg)


In [4]:
retail['Revenue'] = retail['Units'] * retail['Unit_Price']
retail['Total_Cost'] = retail['Units'] * retail['Unit_Cost']
retail['Gross_Profit'] = retail['Revenue'] - retail['Total_Cost']
retail['Margin_Pct'] = 100 * retail['Gross_Profit'] / retail['Revenue'].replace(0, np.nan)
print('Revenue:', retail['Revenue'].sum())
print('Gross profit:', retail['Gross_Profit'].sum())
print('Overall margin:', round(100 * retail['Gross_Profit'].sum() / retail['Revenue'].sum(), 2))


Revenue: 41100
Gross profit: 17260
Overall margin: 42.0


## 4. Let the regions compete

Group revenue and profit by Region, then by Product. Sort by profit. Which region leads? Which product leads? Explain why revenue alone is insufficient—revenue is loud, but profit pays the bills.


In [5]:
regions = retail.groupby('Region')[['Revenue', 'Gross_Profit']].sum().sort_values('Gross_Profit', ascending=False)
products = retail.groupby('Product')[['Revenue', 'Gross_Profit']].sum().sort_values('Gross_Profit', ascending=False)
print(regions)
print(products)


         Revenue  Gross_Profit
Region                        
Kanpur     14400          6010
Lucknow    13400          5710
Delhi      13300          5540
          Revenue  Gross_Profit
Product                        
Planner     17400          6960
Notebook    15500          6200
Pen          8200          4100


## 5. Give everyone a discount

Convert Units and Unit_Price into NumPy arrays. Calculate total revenue after a 5% discount with unchanged unit sales. Compare with current revenue. What assumption could fail? Customers do not always obey spreadsheet assumptions.


In [6]:
units = retail['Units'].to_numpy()
prices = retail['Unit_Price'].to_numpy()
current = np.sum(units * prices)
discounted = np.sum(units * prices * 0.95)
print('Current:', current, '| Discounted:', discounted)
print('Change:', discounted - current)


Current: 41100 | Discounted: 39045.0
Change: -2055.0


## 6. Clean the marketing data

Separate campaigns with missing Spend before comparing paid performance. Retain zero-click campaigns. Why would removing them exaggerate results? Unsuccessful ads still spent real money, sadly.


In [7]:
marketing_review = marketing_raw.loc[marketing_raw['Spend'].isna()].copy()
marketing = marketing_raw.dropna(subset=['Spend']).copy()
print('Usable campaigns:', len(marketing))
print('Campaigns needing spend confirmation:', marketing_review['Campaign_ID'].tolist())


Usable campaigns: 19
Campaigns needing spend confirmation: ['C011']


## 7. Make the marketing acronyms earn their keep

Create CTR_Pct = Clicks/Impressions × 100; CPC = Spend/Clicks; CPL = Spend/Leads; CAC = Spend/Customers; ROAS = Revenue/Spend. Replace zero denominators with NaN. ROAS is a revenue-to-ad-spend ratio, not profit; the acronyms are impressive, but they cannot change accounting.


### From an impression to a customer

Marketing activity narrows like a funnel. Every customer was once an impression, but most impressions never become customers—they saw the ad and continued scrolling.

![Marketing funnel from impressions to revenue](../assets/diagrams/marketing-funnel.svg)


In [8]:
marketing['CTR_Pct'] = 100 * marketing['Clicks'] / marketing['Impressions'].replace(0, np.nan)
marketing['CPC'] = marketing['Spend'] / marketing['Clicks'].replace(0, np.nan)
marketing['CPL'] = marketing['Spend'] / marketing['Leads'].replace(0, np.nan)
marketing['CAC'] = marketing['Spend'] / marketing['Customers'].replace(0, np.nan)
marketing['ROAS'] = marketing['Revenue'] / marketing['Spend'].replace(0, np.nan)
print(marketing[['Campaign_ID', 'CTR_Pct', 'CPC', 'CPL', 'CAC', 'ROAS']].round(2))


   Campaign_ID  CTR_Pct    CPC     CPL      CAC   ROAS
0         C001      5.0   4.80   60.00   300.00   3.33
1         C002      2.0   5.00   66.67   400.00   2.25
2         C003     15.0   1.11   13.33    57.14  14.00
3         C004      5.0   5.00   62.50   333.33   3.00
4         C005      2.0   5.00   71.43   400.00   2.20
5         C006     14.0   1.19   14.71    62.50  12.80
6         C007      0.5  10.00  200.00  1200.00   0.67
7         C008      0.0    NaN     NaN      NaN   0.00
8         C009      2.0   5.00   70.00   466.67   1.71
9         C010      5.0   4.50   56.25   300.00   3.33
11        C012      0.5  10.00  160.00  1000.00   0.80
12        C013      5.0   4.89   61.11   305.56   3.27
13        C014      2.0   5.00   72.22   361.11   2.31
14        C015     15.0   1.11   14.29    62.50  12.67
15        C016      0.5  10.00  157.14   916.67   0.87
16        C017      5.0   4.91   61.36   321.43   3.11
17        C018      2.0   5.00   69.23   391.30   2.28
18        

## 8. Choose a marketing champion

Aggregate Spend, Clicks, Leads, Customers, and Revenue by Channel. Calculate channel ROAS and CAC from totals, not averages of campaign ratios. Which channel leads ROAS? What additional evidence is needed before reallocating budget? The largest number does not automatically get a trophy.


In [9]:
channels = marketing.groupby('Channel')[['Spend', 'Clicks', 'Leads', 'Customers', 'Revenue']].sum()
channels['ROAS'] = channels['Revenue'] / channels['Spend'].replace(0, np.nan)
channels['CAC'] = channels['Spend'] / channels['Customers'].replace(0, np.nan)
print(channels.sort_values('ROAS', ascending=False).round(2))


           Spend  Clicks  Leads  Customers  Revenue   ROAS      CAC
Channel                                                            
Email     9300.0    7950    625        144   114800  12.34    64.58
Search   60500.0   12500   1000        193   193000   3.19   313.47
Social   40500.0    8100    580        101    87500   2.16   400.99
Display  24000.0    1900    110         18    14500   0.60  1333.33


## 9. Write your management recommendation
Write three sentences: one retail finding, one marketing finding, and one limitation or follow-up question. Three useful sentences beat a 30-slide presentation.


### Example interpretation
Kanpur narrowly leads retail revenue and gross profit; Planner leads product profit. A 5% price discount reduces revenue by 5% when units stay constant, but real customers may refuse to follow the spreadsheet. Email leads ROAS among complete campaigns, yet its recorded spend, attribution rules, excluded campaign, and ability to scale require investigation before we hand it the entire marketing budget.


### Expected checkpoints
Use these to check your work, not to negotiate with Python:

- Retail: 20 raw rows, 18 usable orders, one duplicate removed, one incomplete order set aside.
- Retail revenue: INR 41,100; gross profit: INR 17,260; overall margin: 42.00%.
- Discounted revenue: INR 39,045.
- Marketing: 20 raw campaigns; 19 usable campaigns; C011 needs spend confirmation.
- Email ROAS: approximately 12.34; Display ROAS: approximately 0.60.


### Stay connected

If this session helped you, **follow [Zaid Kamil on GitHub](https://github.com/zaid-kamil)** for more learning resources and **[connect on LinkedIn](https://linkedin.com/in/zaid-kamil-94211a40)**. Mention the Amity Lucknow workshop in your connection request.

[![Follow on GitHub](https://img.shields.io/badge/GitHub-Follow_Zaid-181717?logo=github&logoColor=white)](https://github.com/zaid-kamil)
[![Connect on LinkedIn](https://img.shields.io/badge/LinkedIn-Connect-0A66C2)](https://linkedin.com/in/zaid-kamil-94211a40)

You can also star this repository to find the notebooks again.
